# Discriminative Predictive Coding

This notebook demonstrates how to train a simple feedforward network with predictive coding (PC) to discriminate or classify MNIST digits.

In [1]:
%%capture
!pip install torch==2.3.1
!pip install torchvision==0.18.1

In [2]:
!git clone https://github.com/thebuckleylab/jpc.git

%cd jpc
!pip install .

Cloning into 'jpc'...
remote: Enumerating objects: 3119, done.
remote: Counting objects: 100% (418/418), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 3119 (delta 346), reused 356 (delta 312), pack-reused 2701 (from 2)
Receiving objects: 100% (3119/3119), 4.99 MiB | 14.68 MiB/s, done.
Resolving deltas: 100% (2079/2079), done.
/content/jpc
Processing /content/jpc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of optax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of chex to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of lineax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/1

In [3]:
import jpc

import jax
import equinox as eqx
import equinox.nn as nn
import optax

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import warnings
warnings.simplefilter('ignore')  # ignore warnings

/content/jpc/jpc/_core/_init.py:22: SyntaxWarning: invalid escape sequence '\{'
  $\{ f_\ell(\mathbf{z}_{\ell-1}) \}_{\ell=1}^L$ where $f_\ell(\cdot)$ is some
/content/jpc/jpc/_core/_init.py:93: SyntaxWarning: invalid escape sequence '\s'
  $z_i \sim \mathcal{N}(0, \sigma^2)$.
/content/jpc/jpc/_core/_init.py:132: SyntaxWarning: invalid escape sequence '\{'
  $\{ f_{L-\ell+1}(\mathbf{z}_{L-\ell}) \}_{\ell=1}^L$ where $\mathbf{z}_0 = \mathbf{y}$ is
/content/jpc/jpc/_core/_init.py:170: SyntaxWarning: invalid escape sequence '\{'
  """Initialises zero errors for use with ePC $\{ \epsilon_\ell = 0 \}_{l=1}^L$.
/content/jpc/jpc/_core/_grads.py:40: SyntaxWarning: invalid escape sequence '\m'
  with respect to the activities $- ∇_{\mathbf{z}} \mathcal{F}$.
/content/jpc/jpc/_core/_grads.py:110: SyntaxWarning: invalid escape sequence '\m'
  with respect to the activities $∇_{\mathbf{z}} \mathcal{F}$.
/content/jpc/jpc/_core/_grads.py:183: SyntaxWarning: invalid escape sequence '\m'
  with respect

## Hyperparameters

We define some global parameters, including the network architecture, learning rate, batch size, etc.

In [4]:
SEED = 0

INPUT_DIM = 784
WIDTH = 300
DEPTH = 3
OUTPUT_DIM = 10
ACT_FN = "relu"

LEARNING_RATE = 1e-3
BATCH_SIZE = 64
TEST_EVERY = 100
N_TRAIN_ITERS = 300

## Dataset

Some utils to fetch MNIST.

In [5]:
def get_mnist_loaders(batch_size):
    train_data = MNIST(train=True, normalise=True)
    test_data = MNIST(train=False, normalise=True)
    train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )
    test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )
    return train_loader, test_loader


class MNIST(datasets.MNIST):
    def __init__(self, train, normalise=True, save_dir="data"):
        if normalise:
            transform = transforms.Compose(
                [
                    transforms.ToTensor(),
                    transforms.Normalize(
                        mean=(0.1307), std=(0.3081)
                    )
                ]
            )
        else:
            transform = transforms.Compose([transforms.ToTensor()])
        super().__init__(save_dir, download=True, train=train, transform=transform)

    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        img = torch.flatten(img)
        label = one_hot(label)
        return img, label


def one_hot(labels, n_classes=10):
    arr = torch.eye(n_classes)
    return arr[labels]


## Network

For `jpc` to work, we need to provide a network with callable layers. This is easy to do with the PyTorch-like [`nn.Sequential()`](https://docs.kidger.site/equinox/api/nn/sequential/#equinox.nn.Sequential) in [equinox](https://github.com/patrick-kidger/equinox). For example, we can define a ReLU MLP with two hidden layers as follows

In [6]:
key = jax.random.PRNGKey(SEED)
_, *subkeys = jax.random.split(key, 4)
network = [
    nn.Sequential(
        [
            nn.Linear(784, 300, key=subkeys[0]),
            nn.Lambda(jax.nn.relu)
        ],
    ),
    nn.Sequential(
        [
            nn.Linear(300, 300, key=subkeys[1]),
            nn.Lambda(jax.nn.relu)
        ],
    ),
    nn.Linear(300, 10, key=subkeys[2]),
]

You can also use [`jpc.make_mlp()`](https://thebuckleylab.github.io/jpc/api/Utils/#jpc.make_mlp) to define a multi-layer perceptron (MLP) or fully connected network.

In [7]:
network = jpc.make_mlp(
    key,
    input_dim=INPUT_DIM,
    width=WIDTH,
    depth=DEPTH,
    output_dim=OUTPUT_DIM,
    act_fn=ACT_FN,
    use_bias=True
)
print(network)

[Sequential(
  layers=(
    Lambda(fn=Identity()),
    Linear(
      weight=f32[300,784],
      bias=f32[300],
      in_features=784,
      out_features=300,
      use_bias=True
    )
  )
), Sequential(
  layers=(
    Lambda(fn=<PjitFunction of <function relu at 0x7bf6482854e0>>),
    Linear(
      weight=f32[300,300],
      bias=f32[300],
      in_features=300,
      out_features=300,
      use_bias=True
    )
  )
), Sequential(
  layers=(
    Lambda(fn=<PjitFunction of <function relu at 0x7bf6482854e0>>),
    Linear(
      weight=f32[10,300],
      bias=f32[10],
      in_features=300,
      out_features=10,
      use_bias=True
    )
  )
)]


## Train and test

A PC network can be updated in a single line of code with [`jpc.make_pc_step()`](https://thebuckleylab.github.io/jpc/api/Training/#jpc.make_pc_step). Similarly, we can use [`jpc.test_discriminative_pc()`](https://thebuckleylab.github.io/jpc/api/Testing/#jpc.test_discriminative_pc) to compute the network accuracy. Note that these functions are already "jitted" for optimised performance. Below we simply wrap each of these functions in training and test loops, respectively.

In [8]:
def evaluate(model, test_loader):
    avg_test_loss, avg_test_acc = 0, 0
    for _, (img_batch, label_batch) in enumerate(test_loader):
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()

        test_loss, test_acc = jpc.test_discriminative_pc(
            model=model,
            input=img_batch,
            output=label_batch
        )
        avg_test_loss += test_loss
        avg_test_acc += test_acc

    return avg_test_loss / len(test_loader), avg_test_acc / len(test_loader)


def train(
      model,
      lr,
      batch_size,
      test_every,
      n_train_iters
):
    optim = optax.adam(lr)
    opt_state = optim.init(
        (eqx.filter(model, eqx.is_array), None)
    )
    train_loader, test_loader = get_mnist_loaders(batch_size)

    for iter, (img_batch, label_batch) in enumerate(train_loader):
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()

        result = jpc.make_pc_step(
            model=model,
            optim=optim,
            opt_state=opt_state,
            output=label_batch,
            input=img_batch
        )
        model, opt_state = result["model"], result["opt_state"]
        train_loss = result["loss"]
        if ((iter+1) % test_every) == 0:
            _, avg_test_acc = evaluate(model, test_loader)
            print(
                f"Train iter {iter+1}, train loss={train_loss:4f}, "
                f"avg test accuracy={avg_test_acc:4f}"
            )
            if (iter+1) >= n_train_iters:
                break


## Run

In [9]:
train(
    model=network,
    lr=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS
)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:01<00:00, 5812671.90it/s] 


Extracting data/MNIST/raw/train-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 159498.83it/s]


Extracting data/MNIST/raw/train-labels-idx1-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:01<00:00, 1452085.24it/s]


Extracting data/MNIST/raw/t10k-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 13247933.77it/s]


Extracting data/MNIST/raw/t10k-labels-idx1-ubyte.gz to data/MNIST/raw

Train iter 100, train loss=0.084942, avg test accuracy=93.689903
Train iter 200, train loss=0.068984, avg test accuracy=95.102165
Train iter 300, train loss=0.046224, avg test accuracy=96.063705


# Catestropic Forgetting

Our plan is to first train the network on the first 5 digits from MNIST (0-5), evaluate it on the test dataset consisting of the same digits, and then train the model further on the remaining digits (6-9), evaluate it on the remaining digits, and finally, reevalute the final model on the initial 5 digits.

In [16]:
train_data = MNIST(train=True, normalise=True)
train_data.targets

tensor([5, 0, 4,  ..., 5, 6, 8])

In [17]:
from torch.utils.data import Subset

def get_mnist_loaders(batch_size, allowed_digits):
    train_data = MNIST(train=True, normalise=True)
    test_data = MNIST(train=False, normalise=True)

    # Filter indices based on allowed target digits
    train_idx = [i for i, target in enumerate(train_data.targets) if target in allowed_digits]
    test_idx = [i for i, target in enumerate(test_data.targets) if target in allowed_digits]

    # Create Subsets
    train_subset = Subset(train_data, train_idx)
    test_subset = Subset(test_data, test_idx)

    train_loader = DataLoader(
        dataset=train_subset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )
    test_loader = DataLoader(
        dataset=test_subset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )
    return train_loader, test_loader

In [18]:
task1_digits = [0, 1, 2, 3, 4]
task2_digits = [5, 6, 7, 8, 9]

train_loader_t1, test_loader_t1 = get_mnist_loaders(BATCH_SIZE, task1_digits)
train_loader_t2, test_loader_t2 = get_mnist_loaders(BATCH_SIZE, task2_digits)

In [20]:
def train_task(
      model,
      train_loader,
      test_loader,
      lr,
      n_train_iters,
      test_every = 100
):
    optim = optax.adam(lr)
    opt_state = optim.init(
        (eqx.filter(model, eqx.is_array), None)
    )

    for iter, (img_batch, label_batch) in enumerate(train_loader):
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()

        result = jpc.make_pc_step(
            model=model,
            optim=optim,
            opt_state=opt_state,
            output=label_batch,
            input=img_batch
        )
        model, opt_state = result["model"], result["opt_state"]
        train_loss = result["loss"]

        if ((iter+1) % test_every) == 0:
            _, avg_test_acc = evaluate(model, test_loader)
            print(
                f"Train iter {iter+1}, train loss={train_loss:4f}, "
                f"avg test accuracy={avg_test_acc:4f}"
            )
            if (iter+1) >= n_train_iters:
                break
    return model

## Train and Evaluate on Task 1 (Digits 0-5)

In [21]:
network_cf = jpc.make_mlp(
    jax.random.PRNGKey(42),
    input_dim=INPUT_DIM,
    width=WIDTH,
    depth=DEPTH,
    output_dim=OUTPUT_DIM,
    act_fn=ACT_FN,
    use_bias=True
)

print("Trainning on Task 1 (Digits 0-5)")
network_cf = train_task(
    model=network_cf,
    train_loader=train_loader_t1,
    test_loader=test_loader_t1,
    lr=LEARNING_RATE,
    n_train_iters=N_TRAIN_ITERS,
    test_every=TEST_EVERY
)

loss_t1, acc_t1 = evaluate(network_cf, test_loader_t1)
print(f"Task 1: Final Accuracy after training on 5 digits: {acc_t1}")

Trainning on Task 1 (Digits 0-5)
Train iter 100, train loss=0.044940, avg test accuracy=98.222656
Train iter 200, train loss=0.018352, avg test accuracy=98.671875
Train iter 300, train loss=0.031900, avg test accuracy=98.964844
Task 1: Final Accuracy after training on 5 digits: 98.96484375


## Train and Evaluate on Task 2 (Digits 6-9)

In [22]:
print("Trainning on Task 2 (Digits 6-9)")
network_cf = train_task(
    model=network_cf,
    train_loader=train_loader_t2,
    test_loader=test_loader_t2,
    lr=LEARNING_RATE,
    n_train_iters=N_TRAIN_ITERS,
    test_every=TEST_EVERY
)

loss_t2, acc_t2 = evaluate(network_cf, test_loader_t2)
print(f"Task 2: Final Accuracy after training on remaining 5 digits: {acc_t2}")

Trainning on Task 2 (Digits 6-9)
Train iter 100, train loss=0.033359, avg test accuracy=96.479164
Train iter 200, train loss=0.034818, avg test accuracy=97.187500
Train iter 300, train loss=0.018874, avg test accuracy=97.645836
Task 2: Final Accuracy after training on remaining 5 digits: 97.64583587646484


## Re-evaluating on Task 1 (Digits 0-5)

In [23]:
print("Re-evaluating on Task 1 (Digits 0-5)")
loss_t1, acc_t1 = evaluate(network_cf, test_loader_t1)
print(f"Task 1: Final Accuracy of after re-training on 5 digits: {acc_t1}")

Re-evaluating on Task 1 (Digits 0-5)
Task 1: Final Accuracy of after re-training on 5 digits: 0.0
